In [148]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [149]:
df = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']).drop(columns=['timestamp'])  # Load your dataset here
df.head()

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [150]:
df["user_id"] = df["user_id"]-1
df["item_id"] = df["item_id"]-1

In [151]:
positive_ratings = df[df['rating']>= 4].astype(int)
positive_ratings["label"] = 1
positive_ratings = positive_ratings.drop(columns=['rating'])
positive_ratings.head()

,user_id,item_id,label
5,297,473,1
7,252,464,1
11,285,1013,1
12,199,221,1
16,121,386,1


In [152]:
negative_ratings = df[df["rating"] <= 3]
negative_ratings['label'] = 0
negative_ratings = negative_ratings.drop(columns=['rating'])
negative_ratings.head()

,user_id,item_id,label
0,195,241,0
1,185,301,0
2,21,376,0
3,243,50,0
4,165,345,0


In [153]:
implicit_df = pd.concat(
    [positive_ratings, negative_ratings],
    ignore_index=True
)

In [154]:
implicit_df = implicit_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [155]:

train_set, test_set = train_test_split(
    implicit_df,
    test_size=0.2,
    random_state=42
)

In [156]:
class MovieLen100k(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id = self.data.iloc[idx]['user_id']
        item_id = self.data.iloc[idx]['item_id']
        label = self.data.iloc[idx]['label']
        return torch.tensor(user_id, dtype=torch.long), torch.tensor(item_id, dtype=torch.long), torch.tensor(label, dtype=torch.float)

In [157]:
train_data, test_data = MovieLen100k(train_set), MovieLen100k(test_set)
train_loader, test_loader = DataLoader(train_data, batch_size=256, shuffle=True), DataLoader(test_data, batch_size=256, shuffle=False)

In [158]:
class UserTower(nn.Module):
    def __init__(self, num_users, embedding_size, output_size=32):
        super(UserTower, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.fc = nn.Sequential(nn.Linear(embedding_size, 64),
                                nn.ReLU(),
                                nn.Linear(64, output_size))
    def forward(self, user_id):
        user_vector = self.user_embedding(user_id)
        return self.fc(user_vector)

class ItemTower(nn.Module):
    def __init__(self, num_items, embedding_size, output_size=32):
        super(ItemTower, self).__init__() 
        self.item_embedding = nn.Embedding(num_items, embedding_size)      
        self.fc = nn.Sequential(nn.Linear(embedding_size, 64),
                                nn.ReLU(),
                                nn.Linear(64, output_size))
    def forward(self, item_id):
        item_vector = self.item_embedding(item_id)
        return self.fc(item_vector)
    

In [159]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_size):
        super(TwoTowerModel, self).__init__()
        self.user_tower = UserTower(num_users, embedding_size)
        self.item_tower = ItemTower(num_items, embedding_size)


    def forward(self, user_id, item_id):
        user_vector = self.user_tower(user_id)
        item_vector = self.item_tower(item_id)
        return (user_vector * item_vector).sum(1)  # Dot product

In [160]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()

In [161]:
model = TwoTowerModel(num_users=num_users, num_items=num_items, embedding_size=32)

In [162]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss = nn.BCEWithLogitsLoss()

In [163]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for user_id, item_id, label in train_loader:

        optimizer.zero_grad()

        logits = model(user_id, item_id) # logit for BCEWithLogitsLoss, not probability

        l = loss(logits, label)

        l.backward()
        optimizer.step()

        total_loss += l.item()

    avg_train_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}")

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for user_id, item_id, label in test_loader:

            logits = model(user_id, item_id)

            l = loss(logits, label)

            total_loss += l.item()

    avg_test_loss = total_loss / len(test_loader)

    print(f"Test Loss: {avg_test_loss:.4f}")

Epoch 1/10, Train Loss: 0.6283
Test Loss: 0.5947
Epoch 2/10, Train Loss: 0.5761
Test Loss: 0.5864
Epoch 3/10, Train Loss: 0.5624
Test Loss: 0.5809
Epoch 4/10, Train Loss: 0.5535
Test Loss: 0.5727
Epoch 5/10, Train Loss: 0.5492
Test Loss: 0.5778
Epoch 6/10, Train Loss: 0.5470
Test Loss: 0.5764
Epoch 7/10, Train Loss: 0.5425
Test Loss: 0.5844
Epoch 8/10, Train Loss: 0.5399
Test Loss: 0.5806
Epoch 9/10, Train Loss: 0.5378
Test Loss: 0.5806
Epoch 10/10, Train Loss: 0.5354
Test Loss: 0.5853


In [193]:
def recall_at_k(recommended_items, relevant_items, k):
    top_k = recommended_items[:k]

    hits = len(set(top_k) & set(relevant_items)) # find the intersection of recommended and relevant items

    return hits / len(relevant_items)

def precision_at_k(recommended_items, relevant_items, k):
    top_k = recommended_items[:k]

    hits = len(set(top_k) & set(relevant_items))

    return hits / k

import numpy as np

def ndcg_at_k(recommended_items, relevant_items, k):
    top_k = recommended_items[:k]

    dcg = 0.0

    for i, item in enumerate(top_k):
        if item in relevant_items:
            dcg += 1 / np.log2(i + 2) # The i + 2 is because Python starts indexing at 0, while our ranking positions start at 1.

    ideal_hits = min(len(relevant_items), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [194]:
all_items = torch.arange(num_items)

In [195]:
all_recalls = []
all_precisions = []
all_ndcgs = []

for user_id in range(num_users):

    # Get this user's positive items in the test set
    relevant_items = test_set[
        (test_set["user_id"] == user_id) &
        (test_set["label"] == 1)
    ]["item_id"].tolist()

    # Skip users who have no positive test interactions
    if len(relevant_items) == 0:
        continue

    # Convert user ID to tensor
    user_tensor = torch.tensor([user_id], dtype=torch.long)

    # Get user embedding from User Tower
    user_vector = model.user_tower(user_tensor)

    # Get embeddings for ALL items
    item_vectors = model.item_tower(all_items)

    # Calculate score between this user and every item
    scores = user_vector @ item_vectors.T

    # Get Top-K
    top_scores, top_items = torch.topk(scores, k=10)

    # Convert tensor → Python list
    top_k = top_items.squeeze(0).tolist()

    # Calculate metrics
    recall = recall_at_k(
        top_k,
        relevant_items,
        k=10
    )

    precision = precision_at_k(
        top_k,
        relevant_items,
        k=10
    )

    ndcg = ndcg_at_k(
        top_k,
        relevant_items,
        k=10
    )

    # Save results
    all_recalls.append(recall)
    all_precisions.append(precision)
    all_ndcgs.append(ndcg)

In [196]:
print("Mean Recall@10:", np.mean(all_recalls))
print("Mean Precision@10:", np.mean(all_precisions))
print("Mean NDCG@10:", np.mean(all_ndcgs))

Mean Recall@10: 0.003252173301467964
Mean Precision@10: 0.004352557127312296
Mean NDCG@10: 0.0036902797596445615


In this notebook, we're doing implicit feedback (0 for <= 3 rating and 1 for >=4 rating) with Two-towers. But our negative samling is not correct since negative sampling means items that users haven't interacted, not rated bad.